# Installation

## SWI-Prolog

In [1]:
#!sudo apt-get install software-properties-common
!sudo apt-add-repository -y ppa:swi-prolog/stable
!sudo apt-get update
!sudo apt-get install swi-prolog

PPA publishes dbgsym, you may need to include 'main/debug' component
Repository: 'deb https://ppa.launchpadcontent.net/swi-prolog/stable/ubuntu/ jammy main'
Description:
Comprehensive Prolog implementation with extensive libraries and development tools.   Primarily targetted at teaching, RDF processing and web-related tasks, such as creating web services or analysing web content.

Official PPAs for SWI-Prolog. See https://www.swi-prolog.org for further information.
More info: https://launchpad.net/~swi-prolog/+archive/ubuntu/stable
Adding repository.
Adding deb entry to /etc/apt/sources.list.d/swi-prolog-ubuntu-stable-jammy.list
Adding disabled deb-src entry to /etc/apt/sources.list.d/swi-prolog-ubuntu-stable-jammy.list
Adding key to /etc/apt/trusted.gpg.d/swi-prolog-ubuntu-stable.gpg with fingerprint E8B739E3753FF4A12360BA6A4AB3A5F60EA9AEB3
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 h

In [2]:
# test whether swi-prolog is installed and check its version
! swipl --version

SWI-Prolog version 10.0.2 for x86_64-linux


## LangPro & Co.

In [3]:
# get LangPro (nl branch is working with the prove_SICK_NL)
! git clone --single-branch --branch nl https://github.com/kovvalsky/LangPro.git

Cloning into 'LangPro'...
remote: Enumerating objects: 1784, done.
remote: Counting objects: 100% (487/487), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 1784 (delta 438), reused 423 (delta 417), pack-reused 1297 (from 2)
Receiving objects: 100% (1784/1784), 20.15 MiB | 12.71 MiB/s, done.
Resolving deltas: 100% (1126/1126), done.


In [4]:
# get prove_SICK_NL, fr branch
! rm -fr prove_SICK_NL
! git clone --single-branch --branch fr https://github.com/kovvalsky/prove_SICK_NL

Cloning into 'prove_SICK_NL'...
remote: Enumerating objects: 6963, done.
remote: Counting objects: 100% (649/649), done.
remote: Compressing objects: 100% (263/263), done.
remote: Total 6963 (delta 407), reused 604 (delta 374), pack-reused 6314 (from 1)
Receiving objects: 100% (6963/6963), 70.90 MiB | 20.35 MiB/s, done.
Resolving deltas: 100% (6487/6487), done.
Updating files: 100% (6217/6217), done.


In [5]:
# clone french files
! git clone https://github.com/mskandalis/hybrid_nli_fr.git

Cloning into 'hybrid_nli_fr'...
remote: Enumerating objects: 2623, done.
remote: Counting objects: 100% (722/722), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 2623 (delta 656), reused 538 (delta 534), pack-reused 1901 (from 2)
Receiving objects: 100% (2623/2623), 58.76 MiB | 13.53 MiB/s, done.
Resolving deltas: 100% (1584/1584), done.
Updating files: 100% (211/211), done.
Error downloading object: lexical_resources/isa_anto_jdm_knowledge_2013.pl (162b3c9): Smudge error: Error downloading lexical_resources/isa_anto_jdm_knowledge_2013.pl (162b3c9d43893983d0c614edb12b7454b71f0d8ed5c5320bc208a5e2993cd3b1): batch response: This repository exceeded its LFS budget. The account responsible for the budget should increase it to restore access.

Errors logged to '/content/hybrid_nli_fr/.git/lfs/logs/20260902T064513.431618587.log'.
Use `git lfs logs last` to view the log.
error: external filter 'git-lfs filter-process' failed
fatal: lexical_resources/isa_anto_jdm_knowled

In [ ]:
# ! git clone https://github.com/RichardMoot/JDMprolog

# Running

In [6]:
# Add output-only instrumentation to the cloned LangPro source.  The original
# entail_all/0 call and every parameter passed to parList/1 remain unchanged.
from pathlib import Path

entail_file = Path("LangPro/prolog/task/entail.pl")
entail_source = entail_file.read_text(encoding="utf-8-sig")

import_anchor = ":- use_module('../latex/latex_ttterm', [latex_probs_llfs/2]).\n"
import_line = ":- use_module('../xml/xml_output', [write_xml_proof_tree/3]).\n"

helper_anchor = """%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
% solves a problem vs check
solve_problem(PrId_Al, KB-XP, Prem_TTterms, Hypo_TTterms, Prover_Ans, Closed, Status) :-
\tcheck_problem(KB-XP_yes, Prem_TTterms, Hypo_TTterms, 'yes', _, Closed_yes, Status_yes, Br_yes, Tree_yes),
\tcheck_problem(KB-XP_no, Prem_TTterms, Hypo_TTterms, 'no', _,  Closed_no, Status_no, Br_no, Tree_no),
\t% write tableau proofs if needed
"""

helper_replacement = """%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
% Save each generated tableau without changing the inference procedure.
% Separate directories prevent the four tableaux for one problem from
% overwriting one another; the filename itself is exactly the problem ID.
save_generated_tableau(Id-Align, Mode, Tree) :-
\tformat(atom(Directory), 'tableaux/~w/~w', [Align, Mode]),
\tmake_directory_path(Directory),
\tformat(atom(Path), '~w/~w.xml', [Directory, Id]),
\tsetup_call_cleanup(
\t\topen(Path, write, Stream, [encoding(utf8)]),
\t\t(
\t\t\tformat(Stream, '<?xml version="1.0" encoding="UTF-8"?>~n', []),
\t\t\tformat(Stream,
\t\t\t\t'<langpro_tableau_export problem_id="~w" alignment="~w" check="~w">~n',
\t\t\t\t[Id, Align, Mode]),
\t\t\t( nonvar(Tree) ->
\t\t\t\twrite_xml_proof_tree(Stream, Tree, Id)
\t\t\t;\tformat(Stream, '<no_tableau reason="defected_or_unavailable"/>~n', [])
\t\t\t),
\t\t\tformat(Stream, '~n</langpro_tableau_export>~n', [])
\t\t),
\t\tclose(Stream)
\t).

% An output failure must never change the prover's judgment or batch run.
save_generated_tableau_safely(PrId_Al, Mode, Tree) :-
\t( catch(save_generated_tableau(PrId_Al, Mode, Tree), Error,
\t\t(print_message(error, Error), fail)) ->
\t\ttrue
\t;\tformat(user_error,
\t\t\t'Warning: tableau export failed for ~w (~w)~n',
\t\t\t[PrId_Al, Mode])
\t).

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
% solves a problem vs check
solve_problem(PrId_Al, KB-XP, Prem_TTterms, Hypo_TTterms, Prover_Ans, Closed, Status) :-
\t% proof_tree only records the tree object; it does not change rules,
\t% limits, lexical knowledge, alignment, or any other inference parameter.
\t( debMode('proof_tree') -> true; assertz(debMode('proof_tree')) ),
\tcheck_problem(KB-XP_yes, Prem_TTterms, Hypo_TTterms, 'yes', _, Closed_yes, Status_yes, Br_yes, Tree_yes),
\tcheck_problem(KB-XP_no, Prem_TTterms, Hypo_TTterms, 'no', _,  Closed_no, Status_no, Br_no, Tree_no),
\tsave_generated_tableau_safely(PrId_Al, 'yes', Tree_yes),
\tsave_generated_tableau_safely(PrId_Al, 'no', Tree_no),
\t% write tableau proofs if needed
"""

if "save_generated_tableau_safely(PrId_Al, 'yes', Tree_yes)" not in entail_source:
    if entail_source.count(import_anchor) != 1:
        raise RuntimeError("LangPro import anchor not found exactly once; export patch not applied")
    if entail_source.count(helper_anchor) != 1:
        raise RuntimeError("LangPro solve_problem/7 anchor not found exactly once; export patch not applied")
    entail_source = entail_source.replace(import_anchor, import_anchor + import_line, 1)
    entail_source = entail_source.replace(helper_anchor, helper_replacement, 1)
    entail_file.write_text(entail_source, encoding="utf-8")

print("LangPro tableau export enabled: prove_SICK_NL/tableaux/<alignment>/<check>/<problem_id>.xml")


LangPro tableau export enabled: prove_SICK_NL/tableaux/<alignment>/<check>/<problem_id>.xml


In [7]:
! cd prove_SICK_NL && swipl \
    -g "parList([parts([test]), lang(fr), complete_tree, allInt, aall, wn_ant, wn_sim, wn_der, constchck]), entail_all." \
    -t halt \
    -f prolog/main.pl \
    ../LangPro/WNProlog/wn.pl \
    ../hybrid_nli_fr/datasets/langpro/sick_langpro_input_prolog_0_0001.pl \
    ../hybrid_nli_fr/datasets/sentence_ids/sick_fr_id.pl

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
      Un homme chante à une femme
4448: [unknown], unknown,    open, 'Ter',48    XP: []
4449: [unknown], unknown,    open, 'Ter',22    XP: []
4450: [unknown], unknown,    open, 'Ter',66    XP: []
4451: [unknown], unknown,    open, 'Ter',45    XP: []
4452:     [yes], unknown,    open, 'Lim',427   XP: []
      Un rhinocéros broute dans un champ
      Un animal broute dans un champ
4453:      [no], unknown,    open, 'Lim',494   XP: []
      Un rhinocéros broute dans un champ
      Il n'y a pas de rhinocéros dans un champ
4455:      [no],      no,  closed, 'Ter',7     XP: []
4456: [unknown], unknown,    open, 'Ter',388   XP: []
4459:      [no], unknown,    open, 'Ter',82    XP: []
      Il n'y a pas de rhinocéros dans un champ
      Le rhinocéros broute sur l'herbe
4460:     [yes], unknown,    open, 'Ter',174   XP: []
      Un rhinocéros broute dans un champ
      Le rhinocéros broute sur l'herbe
4461:     [yes],

In [8]:
# Create one convenient archive while retaining all individual ID-named files.
import shutil
from pathlib import Path

tableau_files = list(Path("prove_SICK_NL/tableaux").rglob("*.xml"))
if not tableau_files:
    raise RuntimeError("LangPro completed without producing any tableau XML files")

archive_path = shutil.make_archive(
    "sick_fr_langpro_tableaux",
    "zip",
    root_dir="prove_SICK_NL",
    base_dir="tableaux",
)
print(
    f"Saved {len(tableau_files)} individual tableaux under "
    f"prove_SICK_NL/tableaux and archive at {archive_path}"
)

# In Colab, uncomment these two lines if you want the archive to download
# automatically after the run:
# from google.colab import files
# files.download("sick_fr_langpro_tableaux.zip")


Saved 18508 individual tableaux under prove_SICK_NL/tableaux and archive at /content/sick_fr_langpro_tableaux.zip


# Publication-quality PNG visualisation

In [ ]:
# Graphviz is used only after proving has finished.  Installing and invoking it
# cannot affect LangPro's parameters, proof search, or predictions.
import concurrent.futures
import html as html_lib
import os
import re
import shutil
import subprocess
import textwrap
import xml.etree.ElementTree as ET
from pathlib import Path

if shutil.which("dot") is None:
    apt_prefix = ["sudo"] if shutil.which("sudo") else []
    apt_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    subprocess.run(apt_prefix + ["apt-get", "update", "-qq"], check=True, env=apt_env)
    subprocess.run(
        apt_prefix + ["apt-get", "install", "-y", "graphviz", "fonts-dejavu-core"],
        check=True,
        env=apt_env,
    )

DOT_BINARY = shutil.which("dot")
if DOT_BINARY is None:
    raise RuntimeError("Graphviz 'dot' is unavailable; PNG rendering cannot continue")

XML_ROOT = Path("prove_SICK_NL/tableaux")
PNG_ROOT = Path("prove_SICK_NL/tableaux_png")
PNG_DPI = 300
MAX_RENDER_WORKERS = 2  # conservative: complete tableaux can be large


def _normalised_text(element):
    """Return all textual XML content with stable, readable whitespace."""
    if element is None:
        return ""
    return re.sub(r"\s+", " ", "".join(element.itertext())).strip()


def _html_wrapped(text, width=72):
    """Escape text for a Graphviz HTML label and insert deterministic wraps."""
    clean = re.sub(r"\s+", " ", text or "").strip()
    lines = textwrap.wrap(
        clean,
        width=width,
        break_long_words=True,
        break_on_hyphens=False,
    ) or [""]
    return '<BR ALIGN="LEFT"/>'.join(html_lib.escape(line, quote=True) for line in lines)


def _node_label(node):
    """Map one LangPro XML node to a semantically faithful visual card."""
    closer = node.find("closer")
    if closer is not None:
        closer_rule = _normalised_text(closer.find("closer_rule")) or "closure"
        closer_ids = _normalised_text(closer.find("closer_ids"))
        detail = _html_wrapped(f"{closer_rule} · nodes {closer_ids}", 58)
        return (
            '<TABLE BORDER="1" COLOR="#E57373" CELLBORDER="0" CELLSPACING="0" '
            'CELLPADDING="0"><TR><TD BGCOLOR="#FFF1F0" CELLPADDING="9">'
            '<FONT FACE="DejaVu Sans" COLOR="#8A1C1C" POINT-SIZE="10">'
            f'<B>CLOSED BRANCH  ×</B></FONT><BR/><FONT FACE="DejaVu Sans Mono" '
            f'COLOR="#7F1D1D" POINT-SIZE="8">{detail}</FONT></TD></TR></TABLE>'
        )

    if node.find("model") is not None:
        return (
            '<TABLE BORDER="1" COLOR="#4AAE8A" CELLBORDER="0" CELLSPACING="0" '
            'CELLPADDING="0"><TR><TD BGCOLOR="#ECFDF5" CELLPADDING="9">'
            '<FONT FACE="DejaVu Sans" COLOR="#065F46" POINT-SIZE="10">'
            '<B>OPEN BRANCH  ◇</B></FONT><BR/><FONT FACE="DejaVu Sans" '
            'COLOR="#047857" POINT-SIZE="8">Saturated branch / model</FONT>'
            '</TD></TR></TABLE>'
        )

    formula = node.find("formula")
    if formula is None:
        return (
            '<TABLE BORDER="1" COLOR="#CBD5E1" CELLBORDER="0" CELLSPACING="0">'
            '<TR><TD BGCOLOR="#F8FAFC" CELLPADDING="8"><FONT FACE="DejaVu Sans" '
            'COLOR="#475569">No tableau node available</FONT></TD></TR></TABLE>'
        )

    sign = formula.get("sign", "").lower()
    node_id = node.get("id") or node.get("absId") or "?"
    if sign == "true":
        border, header, ink, sign_label = "#5B8FD9", "#EAF2FC", "#173B70", "TRUE"
    else:
        border, header, ink, sign_label = "#E58A4A", "#FFF1E6", "#7C3510", "FALSE"

    llf = _normalised_text(formula.find("llf"))
    mods = [_normalised_text(x) for x in formula.findall("./modList/mod")]
    args = [_normalised_text(x) for x in formula.findall("./argList/arg")]
    source = node.find("source")

    rows = [
        f'<TR><TD BGCOLOR="{header}" ALIGN="LEFT" CELLPADDING="6">'
        f'<FONT FACE="DejaVu Sans" COLOR="{ink}" POINT-SIZE="9">'
        f'<B>NODE {html_lib.escape(str(node_id))}  ·  {sign_label}</B></FONT></TD></TR>'
    ]
    if mods:
        rows.append(
            '<TR><TD ALIGN="LEFT" BGCOLOR="#F4F1FF" CELLPADDING="5">'
            '<FONT FACE="DejaVu Sans" COLOR="#6653A6" POINT-SIZE="8"><B>MOD</B>  '
            f'{_html_wrapped(", ".join(mods), 68)}</FONT></TD></TR>'
        )
    rows.append(
        '<TR><TD ALIGN="LEFT" BGCOLOR="#FFFFFF" CELLPADDING="7">'
        '<FONT FACE="DejaVu Sans Mono" COLOR="#111827" POINT-SIZE="9">'
        f'{_html_wrapped(llf, 72)}</FONT></TD></TR>'
    )
    if args:
        rows.append(
            '<TR><TD ALIGN="LEFT" BGCOLOR="#FFF8D8" CELLPADDING="5">'
            '<FONT FACE="DejaVu Sans" COLOR="#6B5713" POINT-SIZE="8"><B>ARGS</B>  '
            f'{_html_wrapped(", ".join(args), 68)}</FONT></TD></TR>'
        )
    if source is not None:
        rule = source.get("rule") or "rule"
        source_ids = [_normalised_text(x) for x in source.findall("./idList/id")]
        old_constants = [_normalised_text(x) for x in source.findall("./oldConstList/oldConst")]
        provenance = rule
        if source_ids:
            provenance += f" · from [{', '.join(source_ids)}]"
        if old_constants:
            provenance += f" · old constants [{', '.join(old_constants)}]"
        rows.append(
            '<TR><TD ALIGN="LEFT" BGCOLOR="#F8FAFC" CELLPADDING="5">'
            '<FONT FACE="DejaVu Sans Mono" COLOR="#64748B" POINT-SIZE="7">'
            f'{_html_wrapped(provenance, 78)}</FONT></TD></TR>'
        )

    return (
        f'<TABLE BORDER="1" COLOR="{border}" CELLBORDER="0" CELLSPACING="0" '
        f'CELLPADDING="0">{"".join(rows)}</TABLE>'
    )


def _problem_card(tableau, export_root, xml_path):
    problem = tableau.find("problem") if tableau is not None else None
    problem_id = (
        (problem.get("id") if problem is not None else None)
        or export_root.get("problem_id")
        or xml_path.stem
    )
    gold = problem.get("answer", "unknown") if problem is not None else "unknown"
    alignment = export_root.get("alignment") or xml_path.parent.parent.name
    check = export_root.get("check") or xml_path.parent.name
    check_name = "ENTAILMENT (YES)" if check == "yes" else "CONTRADICTION (NO)"
    alignment_name = "ALIGNED LLFs" if alignment == "align" else "NON-ALIGNED LLFs"

    rows = [
        '<TR><TD ALIGN="LEFT" BGCOLOR="#0F172A" CELLPADDING="9">'
        '<FONT FACE="DejaVu Sans" COLOR="#FFFFFF" POINT-SIZE="13">'
        f'<B>LANGPRO TABLEAU  ·  PROBLEM {html_lib.escape(str(problem_id))}</B>'
        '</FONT></TD></TR>',
        '<TR><TD ALIGN="LEFT" BGCOLOR="#E2E8F0" CELLPADDING="6">'
        '<FONT FACE="DejaVu Sans" COLOR="#334155" POINT-SIZE="8">'
        f'<B>{alignment_name}</B>   ·   <B>{check_name}</B>   ·   GOLD: '
        f'{html_lib.escape(str(gold).upper())}</FONT></TD></TR>',
    ]
    if problem is not None:
        for index, premise in enumerate(problem.findall("premise"), start=1):
            pid = premise.get("id") or str(index)
            rows.append(
                '<TR><TD ALIGN="LEFT" BGCOLOR="#F8FAFC" CELLPADDING="5">'
                '<FONT FACE="DejaVu Sans" COLOR="#334155" POINT-SIZE="8">'
                f'<B>P{html_lib.escape(str(pid))}</B>  {_html_wrapped(_normalised_text(premise), 108)}'
                '</FONT></TD></TR>'
            )
        conclusion = problem.find("conclusion")
        if conclusion is not None:
            cid = conclusion.get("id") or "H"
            rows.append(
                '<TR><TD ALIGN="LEFT" BGCOLOR="#FFF7ED" CELLPADDING="5">'
                '<FONT FACE="DejaVu Sans" COLOR="#7C2D12" POINT-SIZE="8">'
                f'<B>H{html_lib.escape(str(cid))}</B>  '
                f'{_html_wrapped(_normalised_text(conclusion), 108)}</FONT></TD></TR>'
            )
    rows.append(
        '<TR><TD ALIGN="LEFT" BGCOLOR="#FFFFFF" CELLPADDING="5">'
        '<FONT FACE="DejaVu Sans" COLOR="#64748B" POINT-SIZE="7">'
        '<B>Legend:</B> TRUE = asserted · FALSE = denied · branch lines preserve XML child order'
        '</FONT></TD></TR>'
    )
    return (
        '<TABLE BORDER="1" COLOR="#94A3B8" CELLBORDER="0" CELLSPACING="0" '
        f'CELLPADDING="0">{"".join(rows)}</TABLE>'
    )


def _xml_to_dot(xml_path):
    export_root = ET.parse(xml_path).getroot()
    tableau = export_root if export_root.tag == "tableau" else export_root.find("tableau")

    dot_nodes = []
    dot_edges = []
    next_node = 0

    def visit(tree_element):
        nonlocal next_node
        graph_id = f"n{next_node}"
        next_node += 1
        xml_node = tree_element.find("node")
        label = _node_label(xml_node) if xml_node is not None else _node_label(ET.Element("node"))
        dot_nodes.append(f"  {graph_id} [label=<{label}>];")
        subtrees = tree_element.find("subTrees")
        if subtrees is not None:
            for child in subtrees.findall("tree"):
                child_id = visit(child)
                dot_edges.append(f"  {graph_id} -> {child_id};")
        return graph_id

    root_tree = tableau.find("tree") if tableau is not None else None
    if root_tree is not None:
        root_id = visit(root_tree)
        status_node = ""
    else:
        root_id = "status"
        status_node = (
            '  status [label=<<TABLE BORDER="1" COLOR="#CBD5E1" CELLBORDER="0" '
            'CELLSPACING="0"><TR><TD BGCOLOR="#F8FAFC" CELLPADDING="10">'
            '<FONT FACE="DejaVu Sans" COLOR="#475569"><B>NO TABLEAU GENERATED</B>'
            '</FONT></TD></TR></TABLE>>];'
        )

    card = _problem_card(tableau, export_root, xml_path)
    lines = [
        "digraph LangProTableau {",
        '  graph [rankdir="TB", bgcolor="#FFFFFF", pad="0.32", nodesep="0.24",',
        f'         ranksep="0.46", splines="spline", outputorder="edgesfirst", dpi="{PNG_DPI}",',
        '         charset="UTF-8", ordering="out"];',
        '  node [shape="plain", fontname="DejaVu Sans"];',
        '  edge [arrowhead="none", color="#94A3B8", penwidth="1.15"];',
        f'  metadata [label=<{card}>];',
    ]
    if status_node:
        lines.append(status_node)
    lines.extend(dot_nodes)
    lines.append(f'  metadata -> {root_id} [style="invis", weight="100", minlen="1"];')
    lines.extend(dot_edges)
    lines.append("}")
    return "\n".join(lines) + "\n"


def _render_one_tableau(xml_path):
    relative = xml_path.relative_to(XML_ROOT).with_suffix(".png")
    png_path = PNG_ROOT / relative
    png_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = png_path.with_suffix(".png.tmp")
    dot_source = _xml_to_dot(xml_path)
    try:
        completed = subprocess.run(
            [DOT_BINARY, "-Tpng", "-o", str(temporary_path)],
            input=dot_source.encode("utf-8"),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=600,
            check=False,
        )
        if completed.returncode != 0:
            diagnostic = completed.stderr.decode("utf-8", errors="replace").strip()
            raise RuntimeError(f"Graphviz failed for {xml_path}: {diagnostic}")
        if temporary_path.read_bytes()[:8] != b"\x89PNG\r\n\x1a\n":
            raise RuntimeError(f"Graphviz did not create a valid PNG for {xml_path}")
        temporary_path.replace(png_path)
        return png_path
    finally:
        temporary_path.unlink(missing_ok=True)


xml_tableaux = sorted(XML_ROOT.rglob("*.xml"))
if not xml_tableaux:
    raise RuntimeError(f"No tableau XML files found under {XML_ROOT}")

render_failures = []
rendered_pngs = []
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_RENDER_WORKERS) as executor:
    future_to_xml = {executor.submit(_render_one_tableau, path): path for path in xml_tableaux}
    for completed_count, future in enumerate(concurrent.futures.as_completed(future_to_xml), start=1):
        source_xml = future_to_xml[future]
        try:
            rendered_pngs.append(future.result())
        except Exception as error:
            render_failures.append(f"{source_xml}: {error}")
        if completed_count % 25 == 0 or completed_count == len(xml_tableaux):
            print(f"Rendered {completed_count}/{len(xml_tableaux)} tableau PNGs")

# Defensive cleanup: only final .png files belong in the deliverable archive.
for temporary_residue in PNG_ROOT.rglob("*.tmp"):
    temporary_residue.unlink(missing_ok=True)

if render_failures:
    failure_report = Path("langpro_png_render_failures.txt")
    failure_report.write_text("\n".join(render_failures) + "\n", encoding="utf-8")
    raise RuntimeError(
        f"{len(render_failures)} tableau(s) could not be rendered; see {failure_report}"
    )

png_archive_path = shutil.make_archive(
    "sick_fr_langpro_tableaux_png",
    "zip",
    root_dir="prove_SICK_NL",
    base_dir="tableaux_png",
)
print(
    f"Saved {len(rendered_pngs)} publication-quality PNGs under {PNG_ROOT} "
    f"and archive at {png_archive_path}"
)

# Optional automatic Colab download:
# from google.colab import files
# files.download("sick_fr_langpro_tableaux_png.zip")


Rendered 25/18508 tableau PNGs
Rendered 50/18508 tableau PNGs
Rendered 75/18508 tableau PNGs
Rendered 100/18508 tableau PNGs
Rendered 125/18508 tableau PNGs
Rendered 150/18508 tableau PNGs
Rendered 175/18508 tableau PNGs
Rendered 200/18508 tableau PNGs
Rendered 225/18508 tableau PNGs
Rendered 250/18508 tableau PNGs
Rendered 275/18508 tableau PNGs
Rendered 300/18508 tableau PNGs
Rendered 325/18508 tableau PNGs
Rendered 350/18508 tableau PNGs
Rendered 375/18508 tableau PNGs
Rendered 400/18508 tableau PNGs
Rendered 425/18508 tableau PNGs
